# Deep Past Challenge — Fine-Tune mattiaangeli's ByT5 for 40+

### Strategy: Fine-tune the STRONG pretrained model, don't start from scratch

**Why this works:**
- `mattiaangeli/byt5-akkadian-mbr-v2` already scores 35.5 — it KNOWS Akkadian
- Scratch training from `google/byt5-large` needs weeks and may never match
- Fine-tuning adapts the model to OUR V3 preprocessing + training data
- Creating 2 variants with different data = ensemble diversity

### Expected Score Path
| Component | Score |
|-----------|-------|
| mattiaangeli baseline | 35.5 |
| + V3 preprocessing (inference) | 36-36.5 |
| + Fine-tuned variant A (all data) | 37-38 |
| + Ensemble (variant A + mattiaangeli + variant B) | 39-41 |
| + Geo-mean MBR | 40-42 |

### How to Use
1. **Run 1**: Set `VARIANT = 'A'` → trains on all data → upload as model version 1
2. **Run 2**: Set `VARIANT = 'B'` → trains on high-quality only → upload as model version 2
3. Use `kaggle_inference_v2.ipynb` with both variants + original mattiaangeli

### Kaggle Setup
- **GPU**: T4x2 (or P100, A100)
- **Internet**: ON (first run) or attach model as dataset
- **Input**: Your processed training data + mattiaangeli's model
- **Time**: ~2-3 hours per variant on T4

In [ ]:
!pip install -q sacrebleu transformers[torch] accelerate

import os, json, math, gc, time, shutil, re, unicodedata, copy
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from torch.utils.data import Dataset
import sacrebleu

print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} | VRAM: {vram:.1f}GB")
else:
    print("WARNING: No GPU detected!")

start_time = time.time()

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# ========= CHANGE THIS BETWEEN RUNS =========
VARIANT = 'A'   # 'A' = all data (broad), 'B' = high-quality only (focused)
# =============================================

# --- Base Model (ALWAYS mattiaangeli, NEVER google/byt5-large) ---
BASE_MODEL = None
for p in [
    "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1",
    "/kaggle/input/byt5-akkadian-mbr-v2/pytorch/default/1",
    "/kaggle/input/byt5-akkadian-mbr-v2",
    "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr/pytorch/default/6",
]:
    if Path(p).exists() and (Path(p) / "config.json").exists():
        BASE_MODEL = p
        break

if BASE_MODEL is None:
    # Fallback: download from HuggingFace (needs internet)
    BASE_MODEL = "mattiaangeli/byt5-akkadian-mbr-v2"
    print("WARNING: No local model found, will download from HuggingFace")

print(f"Base model: {BASE_MODEL}")

# --- Data Paths ---
TRAIN_FILE = "/kaggle/input/akkadian-processed-data/train_split.csv"
VAL_FILE = "/kaggle/input/akkadian-processed-data/val_split.csv"
if not Path(TRAIN_FILE).exists():
    TRAIN_FILE = "../data_processed/combined/train_split.csv"
    VAL_FILE = "../data_processed/combined/val_split.csv"

# --- Output ---
OUTPUT_DIR = Path("/kaggle/working/model") if Path("/kaggle").exists() else Path(f"../models/finetune_{VARIANT}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Sequence Lengths ---
MAX_SRC_LEN = 512
MAX_TGT_LEN = 512
PREFIX = "translate Akkadian to English: "
SEED = 42
MAX_BYTE_FILTER = 1024
NUM_EVAL_SAMPLES = 100

# --- GPU-aware batch config ---
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    IS_HIGH_END = 'H100' in gpu_name or 'A100' in gpu_name
    if IS_HIGH_END:
        BATCH_SIZE, GRAD_ACCUM = 4, 4
        USE_BF16, USE_FP16 = True, False
    elif vram >= 14:  # T4
        BATCH_SIZE, GRAD_ACCUM = 2, 8
        USE_BF16, USE_FP16 = False, False  # FP32 for ByT5!
    else:
        BATCH_SIZE, GRAD_ACCUM = 1, 16
        USE_BF16, USE_FP16 = False, False
else:
    BATCH_SIZE, GRAD_ACCUM = 1, 16
    USE_BF16, USE_FP16 = False, False

# === VARIANT-SPECIFIC TRAINING CONFIG ===
# LESSON LEARNED: lr=5e-6 was 10-50x too low. Model barely moved.
# Need lr=5e-5 to 1e-4 for meaningful fine-tuning of 582M param model.
# More epochs + frequent eval to catch the sweet spot before overfitting.
if VARIANT == 'A':
    # Variant A: Fine-tune on ALL data with aggressive LR
    PHASES = [
        {
            "name": "Phase 1: All Data (aggressive fine-tune)",
            "epochs": 5,
            "lr": 5e-5,          # 10x higher - actually moves the weights
            "warmup_ratio": 0.06,
            "label_smoothing": 0.1,
            "weight_decay": 0.01,
            "sources": None,     # All training data (~13K pairs)
            "lr_scheduler_type": "cosine",
            "eval_steps": 200,   # Evaluate every 200 steps to catch sweet spot
        },
        {
            "name": "Phase 2: Strategy_a Polish",
            "epochs": 5,
            "lr": 1e-5,          # Still 10x higher than before
            "warmup_ratio": 0.1,
            "label_smoothing": 0.0,
            "weight_decay": 0.005,
            "sources": ["strategy_a"],
            "lr_scheduler_type": "cosine",
            "eval_steps": 50,
        },
    ]
elif VARIANT == 'B':
    # Variant B: High-quality data only, different LR = different errors for ensemble
    PHASES = [
        {
            "name": "Phase 1: High-Quality Sentence Pairs",
            "epochs": 8,
            "lr": 3e-5,
            "warmup_ratio": 0.08,
            "label_smoothing": 0.1,
            "weight_decay": 0.01,
            "sources": ["strategy_a", "strategy_b_aligned", "strategy_b_short"],
            "lr_scheduler_type": "cosine",
            "eval_steps": 100,
        },
        {
            "name": "Phase 2: Strategy_a Final Polish",
            "epochs": 5,
            "lr": 5e-6,
            "warmup_ratio": 0.1,
            "label_smoothing": 0.0,
            "weight_decay": 0.005,
            "sources": ["strategy_a"],
            "lr_scheduler_type": "cosine",
            "eval_steps": 30,
        },
    ]
else:
    raise ValueError(f"Unknown variant: {VARIANT}. Use A or B.")

print(f"\n=== VARIANT {VARIANT} ===")
print(f"Effective batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"Precision: {'BF16' if USE_BF16 else 'FP32'}")
for i, p in enumerate(PHASES):
    print(f"  Phase {i+1}: {p['name']} | {p['epochs']}ep | lr={p['lr']}")

In [ ]:
# ============================================================
# V3 PREPROCESSING - matches competition test data format
# ============================================================

def v3_normalize_transliteration(text):
    """Normalize transliteration to match v3 test format."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFC', text)
    text = text.replace('\u1e2b', 'h').replace('\u1e2a', 'H')
    text = text.replace('<big_gap>', '<gap>')
    text = re.sub(r'\.{3,}|\u2026+', '<gap>', text)
    text = re.sub(r'(?<![a-zA-Z\-])\bx\b(?![a-zA-Z\-])', '<gap>', text)
    text = re.sub(r'(<gap>[\s\-]*){2,}', '<gap> ', text)
    # Remove brackets (protect gaps)
    text = text.replace('<gap>', '\x00G\x00')
    text = re.sub(r'\[([^\]]*?)\]', r'\1', text)
    text = text.replace('\x00G\x00', '<gap>')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def v3_normalize_translation(text):
    """Normalize translation to match v3 test format."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\bPN\b', '<gap>', text)
    text = text.replace('\u1e2b', 'h').replace('\u1e2a', 'H')
    text = text.replace('<big_gap>', '<gap>')
    text = re.sub(r'\.{3,}|\u2026+', '<gap>', text)
    text = re.sub(r'(<gap>[\s\-]*){2,}', '<gap> ', text)
    # Remove annotations
    text = re.sub(r'\s*\((fem|plur|pl|sing|singular|plural)\.?\s*\w*\)', '', text, flags=re.I)
    text = re.sub(r'\s*\([?!]\)', '', text)
    # Fractions -> Unicode (BEFORE any char removal)
    for frac, uni in {'1/3':'\u2153','2/3':'\u2154','1/6':'\u2159','5/6':'\u215a',
                      '1/2':'\u00bd','1/4':'\u00bc','3/4':'\u00be'}.items():
        text = text.replace(frac, uni)
    text = re.sub(r'\b0\.5\b', '\u00bd', text)
    text = re.sub(r'(\d+)\.5\b', lambda m: f"{m.group(1)}\u00bd", text)
    text = re.sub(r'\b0\.25\b', '\u00bc', text)
    text = re.sub(r'\b0\.75\b', '\u00be', text)
    # Remove brackets (protect gaps)
    text = text.replace('<gap>', '\x00G\x00')
    text = re.sub(r'\[([^\]]*?)\]', r'\1', text)
    text = text.replace('\x00G\x00', '<gap>')
    # Standardize quotes
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    # Gap spacing
    text = re.sub(r'(?<![- ])<gap>(?![- ])', ' <gap> ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Self-test
_t1 = v3_normalize_transliteration("um-ma \u1e2aa-nu-um-ma [x] ... a-na")
assert '\u1e2b' not in _t1.lower() and '<big_gap>' not in _t1
_t2 = v3_normalize_translation("Seal of PN son of 1/3 mina (plur)")
assert 'PN' not in _t2 and '\u2153' in _t2 and '(plur)' not in _t2
print("V3 Preprocessing OK")
print(f"  Translit: {_t1}")
print(f"  Translat: {_t2}")

In [ ]:
# ============================================================
# LOAD & PREPROCESS DATA
# ============================================================
print(f"Loading: {TRAIN_FILE}")
train_data = pd.read_csv(TRAIN_FILE).dropna(subset=['transliteration', 'translation'])
val_data = pd.read_csv(VAL_FILE).dropna(subset=['transliteration', 'translation'])
print(f"Raw: {len(train_data)} train | {len(val_data)} val")

if 'source' in train_data.columns:
    print("\nSources:")
    print(train_data['source'].value_counts().to_string())

print("\nApplying v3 normalization...")
train_data['transliteration'] = train_data['transliteration'].apply(v3_normalize_transliteration)
train_data['translation'] = train_data['translation'].apply(v3_normalize_translation)
val_data['transliteration'] = val_data['transliteration'].apply(v3_normalize_transliteration)
val_data['translation'] = val_data['translation'].apply(v3_normalize_translation)

# Filter too-short and too-long
train_data = train_data[train_data['transliteration'].str.len() > 5]
train_data = train_data[train_data['translation'].str.len() > 5]
src_bytes = train_data['transliteration'].str.encode('utf-8').str.len()
tgt_bytes = train_data['translation'].str.encode('utf-8').str.len()
train_data = train_data[(src_bytes <= MAX_BYTE_FILTER) & (tgt_bytes <= MAX_BYTE_FILTER)]
train_data = train_data.reset_index(drop=True)
print(f"After filter: {len(train_data)} train")

print("\nSample pairs:")
for i in range(min(3, len(train_data))):
    r = train_data.iloc[i]
    print(f"  SRC: {r['transliteration'][:80]}")
    print(f"  TGT: {r['translation'][:80]}\n")

In [ ]:
# ============================================================
# DATASET & EVALUATION
# ============================================================

class Seq2SeqDataset(Dataset):
    def __init__(self, df, tokenizer, max_src=512, max_tgt=512, prefix=""):
        self.samples = []
        t0 = time.time()
        for _, row in df.iterrows():
            src = prefix + str(row['transliteration'])
            tgt = str(row['translation'])
            src_enc = tokenizer(src, max_length=max_src, truncation=True)
            tgt_enc = tokenizer(text_target=tgt, max_length=max_tgt, truncation=True)
            self.samples.append({
                'input_ids': src_enc['input_ids'],
                'attention_mask': src_enc['attention_mask'],
                'labels': tgt_enc['input_ids'],
            })
        print(f"  Tokenized {len(self.samples)} in {time.time()-t0:.1f}s")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def filter_by_source(data, sources):
    """Filter training data by source column."""
    if sources is None:
        return data
    if 'source' not in data.columns:
        print("  WARNING: No source column, using all data")
        return data
    filtered = data[data['source'].isin(sources)]
    print(f"  Filtered: {len(filtered)} rows from {sources}")
    if len(filtered) < 50:
        print(f"  WARNING: Only {len(filtered)} samples! Check source names.")
        print(f"  Available: {data['source'].unique().tolist()}")
    return filtered


def evaluate_model(model, tokenizer, val_ds, max_samples=100, num_beams=5):
    """Evaluate model on validation set using competition metric."""
    model.eval()
    device = next(model.parameters()).device
    preds, refs = [], []
    n = min(max_samples, len(val_ds))
    print(f"  Evaluating {n} samples...", end=" ", flush=True)
    t0 = time.time()
    for i in range(n):
        s = val_ds[i]
        ids = torch.tensor([s['input_ids']], device=device)
        mask = torch.tensor([s['attention_mask']], device=device)
        with torch.no_grad():
            out = model.generate(
                input_ids=ids, attention_mask=mask,
                max_new_tokens=256, num_beams=num_beams,
                length_penalty=1.3, repetition_penalty=1.2,
            )
        pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        lab = [t for t in s['labels'] if t >= 0 and t != tokenizer.pad_token_id]
        lab = [min(t, 258) for t in lab]
        ref = tokenizer.decode(lab, skip_special_tokens=True).strip()
        preds.append(pred)
        refs.append(ref)

    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=2).score
    geo = math.sqrt(bleu * chrf) if bleu > 0 and chrf > 0 else 0.0
    print(f"done in {time.time()-t0:.0f}s")
    print(f"  BLEU={bleu:.2f} | chrF++={chrf:.2f} | GeoMean={geo:.2f}")
    for j in range(min(3, n)):
        print(f"  [{j}] pred: {preds[j][:80]}")
        print(f"       ref:  {refs[j][:80]}")
    return {'bleu': round(bleu, 2), 'chrf_pp': round(chrf, 2), 'geo_mean': round(geo, 2)}

print("Dataset and evaluation defined.")

In [ ]:
# ============================================================
# LOAD MODEL & ESTABLISH BASELINE
# ============================================================
print(f"Loading {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
)
if torch.cuda.is_available():
    model = model.cuda()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params/1e6:.0f}M | dtype: {next(model.parameters()).dtype}")

print("\nTokenizing validation set...")
val_ds = Seq2SeqDataset(val_data, tokenizer, MAX_SRC_LEN, MAX_TGT_LEN, PREFIX)

# CRITICAL: Establish baseline BEFORE training
# This tells us if our fine-tuning actually helps
print("\nBaseline evaluation (before fine-tuning):")
baseline = evaluate_model(model, tokenizer, val_ds, NUM_EVAL_SAMPLES, num_beams=5)
print(f"\nBaseline GeoMean = {baseline['geo_mean']}")
print(f"If fine-tuning hurts, we rollback to this.")

# Save baseline state for rollback
best_geo = baseline["geo_mean"]
best_phase = 'baseline'
best_state = copy.deepcopy(model.state_dict())

print(f"\nElapsed: {(time.time()-start_time)/60:.1f} min")

In [ ]:
# ============================================================
# FINE-TUNING (2-phase curriculum)
# ============================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, label_pad_token_id=-100
)

all_results = {"baseline": baseline}

for phase_idx, cfg in enumerate(PHASES):
    phase_num = phase_idx + 1
    phase_name = cfg['name']

    print(f"\n{'='*60}")
    print(f"{phase_name}")
    print(f"{'='*60}")

    phase_data = filter_by_source(train_data, cfg['sources'])
    if len(phase_data) < 10:
        print(f"Skipping - only {len(phase_data)} samples")
        continue

    print(f"Data: {len(phase_data)} | Epochs: {cfg['epochs']} | LR: {cfg['lr']}")
    train_ds = Seq2SeqDataset(phase_data, tokenizer, MAX_SRC_LEN, MAX_TGT_LEN, PREFIX)

    total_steps = max(1, len(phase_data) * cfg['epochs'] // (BATCH_SIZE * GRAD_ACCUM))
    warmup_steps = int(total_steps * cfg['warmup_ratio'])
    log_steps = max(10, total_steps // 20)
    phase_dir = OUTPUT_DIR / f"phase{phase_num}"

    # Use per-step eval if configured, otherwise per-epoch
    eval_steps = cfg.get('eval_steps', None)
    eval_strat = 'steps' if eval_steps else 'epoch'
    save_strat = 'steps' if eval_steps else 'epoch'
    lr_sched = cfg.get('lr_scheduler_type', 'linear')

    args = Seq2SeqTrainingArguments(
        output_dir=str(phase_dir),
        num_train_epochs=cfg['epochs'],
        learning_rate=cfg['lr'],
        lr_scheduler_type=lr_sched,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        weight_decay=cfg['weight_decay'],
        label_smoothing_factor=cfg['label_smoothing'],
        bf16=USE_BF16,
        fp16=USE_FP16,
        predict_with_generate=False,
        eval_strategy=eval_strat,
        eval_steps=eval_steps,
        save_strategy=save_strat,
        save_steps=eval_steps,
        save_total_limit=3,
        save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=log_steps,
        report_to='none',
        seed=SEED,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
    )

    print(f"Steps: ~{total_steps} | Warmup: {warmup_steps}")
    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=data_collator,
    )

    trainer.train()

    print(f"\nEvaluating Phase {phase_num}...")
    results = evaluate_model(model, tokenizer, val_ds, NUM_EVAL_SAMPLES, num_beams=5)
    all_results[phase_name] = results

    if results['geo_mean'] > best_geo:
        best_geo = results['geo_mean']
        best_phase = phase_name
        best_state = copy.deepcopy(model.state_dict())
        best_dir = OUTPUT_DIR / 'best'
        best_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(best_dir), safe_serialization=True)
        tokenizer.save_pretrained(str(best_dir))
        print(f"  NEW BEST! GeoMean={best_geo:.2f}")
    else:
        print(f"  No improvement (best: {best_geo:.2f} from {best_phase})")
        # ROLLBACK to best state - don't let bad phase persist
        print(f"  Rolling back to best model...")
        model.load_state_dict(best_state)

    del trainer, train_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if phase_dir.exists():
        shutil.rmtree(phase_dir, ignore_errors=True)
    print(f"Phase {phase_num} done. Elapsed: {(time.time()-start_time)/60:.1f} min")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"Best: {best_phase} (GeoMean={best_geo:.2f})")
print(f"Baseline was: {baseline['geo_mean']:.2f}")
improvement = best_geo - baseline['geo_mean']
print(f"Improvement: {improvement:+.2f}")
if improvement < 0:
    print("WARNING: Fine-tuning HURT the model! Saving baseline instead.")
print(f"Total: {(time.time()-start_time)/60:.1f} min")
print(f"{'='*60}")

In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================
final_dir = OUTPUT_DIR / 'final'
final_dir.mkdir(parents=True, exist_ok=True)

# Always save the BEST model (might be baseline if fine-tuning hurt)
if best_state is not None:
    print(f"Saving best model from: {best_phase}")
    model.load_state_dict(best_state)

model.save_pretrained(str(final_dir), safe_serialization=True)
tokenizer.save_pretrained(str(final_dir))

with open(str(final_dir / 'training_results.json'), 'w') as f:
    json.dump({
        'variant': VARIANT,
        'results': all_results,
        'best_phase': best_phase,
        'best_geo_mean': best_geo,
        'baseline_geo_mean': baseline['geo_mean'],
        'base_model': BASE_MODEL,
        'improvement': best_geo - baseline['geo_mean'],
    }, f, indent=2)

print(f"\nFinal model saved to: {final_dir}")
print(f"\nAll Results:")
for name, r in all_results.items():
    marker = " <- BEST" if name == best_phase else ""
    print(f"  {name}: Geo={r['geo_mean']} | BLEU={r['bleu']} | chrF++={r['chrf_pp']}{marker}")

model_size = sum(f.stat().st_size for f in final_dir.rglob("*") if f.is_file()) / 1e9
print(f"\nModel size: {model_size:.2f} GB")
print(f"\nNext steps:")
print(f"  1. Upload to Kaggle Models as version")
if VARIANT == 'A':
    print(f"  2. Change VARIANT='B' and run again for ensemble diversity")
elif VARIANT == 'B':
    print(f"  2. Use kaggle_inference_v2.ipynb with both variants + original mattiaangeli")
print(f"\nTotal: {(time.time()-start_time)/60:.1f} min")

In [ ]:
# ============================================================
# QUICK TEST
# ============================================================
model.eval()
device = next(model.parameters()).device

test_inputs = [
    "um-ma \u0161a-lim-a-\u0161\u00f9r-ma a-na \u0161u-be-lim-ma",
    "1 T\u00daG \u0161a q\u00e1-tim i-tur\u2084-DINGIR il\u2085-q\u00e9",
    "KI\u0160IB ma-nu-ba-l\u00fam-a-\u0161\u00f9r DUMU \u1e63\u00ed-l\u00e1-{d}IM",
    "a-na a-lim{ki} i-l\u00e1-ak-ma K\u00d9.BABBAR \u00fa-\u0161\u00e9-bi\u2084-lam",
]

print(f"Sample translations (Variant {VARIANT}):\n")
for t in test_inputs:
    t_norm = v3_normalize_transliteration(t)
    enc = tokenizer(
        PREFIX + t_norm, max_length=MAX_SRC_LEN,
        truncation=True, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=256, num_beams=5, length_penalty=1.3)
    trans = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"  AKK: {t}")
    print(f"  ENG: {trans}\n")

print("Done! Upload model to Kaggle Models, then use kaggle_inference_v2.ipynb.")